# Comprehensive Counterfactual Dynamical Analysis for All AKOrN Models

This notebook performs comprehensive dynamical analysis on all 15 cases (opbs_0 through opbs_14) from the parameter sweep using the `AKOrNDynamicalAnalyzer` class. We analyze energy dynamics, temporal evolution, convergence properties, and network characteristics across all gamma and T value combinations.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os
import json
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Add project root to path (since we're in notebooks/)
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Setup imports
from source.models.classification.my_knet import MyAKOrN
from source.models.classification.analysis_utils import AKOrNDynamicalAnalyzer, AKOrNStaticAnalyzer
from source.kuramoto_network_metrics import (
    compute_all_metrics, 
    spectral_metrics, 
    strength_metrics,
    community_metrics,
    path_metrics,
    graph_from_K
)
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
import networkx as nx

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Project root: {project_root}")

## Load All 15 Sweep Cases

We load all models from sweep_20250708_*.opbs_0 through sweep_20250708_*.opbs_14 for comprehensive analysis.

In [ ]:
# Generate all 15 cases (opbs_0 through opbs_14)
all_cases = []
for i in range(15):
    all_cases.append({
        "dir": f"sweep_20250708_581{384+i}.opbs_{i}",
        "index": i,
        "name": f"Case {i}"
    })

print(f"Generated {len(all_cases)} cases for comprehensive analysis:")
for case in all_cases:
    print(f"  {case['name']}: {case['dir']}")

In [ ]:
def load_model_from_sweep(sweep_dir):
    """Load a trained model from sweep results."""
    results_dir = project_root / "results"
    
    # Load config
    config_path = results_dir / sweep_dir / "parameters.json"
    if not config_path.exists():
        print(f"Config not found for {sweep_dir}")
        return None
    
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    # Check if model exists
    model_path = results_dir / sweep_dir / "my_akorn_cifar10_final.pth"
    if not model_path.exists():
        print(f"Model not found for {sweep_dir}")
        return None
    
    try:
        # Create model
        model = MyAKOrN(
            n=config['n'],
            ch=config['ch'], 
            out_classes=config['num_classes'],
            L=config['L'],
            T=config['T'],
            J=config['J'],
            J_bias=config['J_bias'],
            ksizes=config['ksizes'],
            ro_ksize=config['ro_ksize'],
            ro_N=config['ro_N'],
            norm=config['norm'],
            c_norm=config['c_norm'],
            gamma=config['gamma'],
            use_omega=config['use_omega'],
            init_omg=config['init_omg'],
            global_omg=config['global_omg'],
            learn_omg=config['learn_omg'],
            ensemble=config['ensemble']
        ).to(device)
        
        # Load weights
        checkpoint = torch.load(model_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        
        print(f"Successfully loaded {sweep_dir} (gamma={config['gamma']}, T={config['T']})")
        return model, config
        
    except Exception as e:
        print(f"Error loading {sweep_dir}: {e}")
        return None

In [ ]:
# Load all 15 models
loaded_models = {}
parameter_summary = []

for case in all_cases:
    result = load_model_from_sweep(case["dir"])
    if result is not None:
        model, config = result
        loaded_models[case["name"]] = {
            "model": model,
            "config": config,
            "gamma": config["gamma"],
            "T": config["T"],
            "sweep_dir": case["dir"],
            "index": case["index"]
        }
        
        # Add to parameter summary
        parameter_summary.append({
            "case": case["name"],
            "index": case["index"],
            "gamma": config["gamma"],
            "T": config["T"],
            "sweep_dir": case["dir"]
        })

print(f"\nSuccessfully loaded {len(loaded_models)} models for comprehensive analysis")

# Create parameter summary DataFrame
param_df = pd.DataFrame(parameter_summary)
print("\nParameter Summary:")
print(param_df.to_string(index=False))

## Parameter Space Visualization

Let's visualize the parameter space coverage across all 15 cases.

In [ ]:
# Visualize parameter space
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Plot 1: Gamma vs T scatter
scatter = axes[0].scatter(param_df['gamma'], param_df['T'], 
                         c=param_df['index'], cmap='viridis', 
                         s=100, alpha=0.7, edgecolors='black')
axes[0].set_xlabel('Gamma', fontsize=12)
axes[0].set_ylabel('T', fontsize=12)
axes[0].set_title('Parameter Space Coverage', fontsize=14)
axes[0].set_xscale('log')
axes[0].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[0], label='Case Index')

# Add case labels
for _, row in param_df.iterrows():
    axes[0].annotate(f"{row['index']}", 
                    (row['gamma'], row['T']), 
                    xytext=(5, 5), textcoords='offset points', 
                    fontsize=9, color='white', weight='bold')

# Plot 2: Distribution of gamma values
gamma_counts = param_df['gamma'].value_counts().sort_index()
axes[1].bar(range(len(gamma_counts)), gamma_counts.values, 
           alpha=0.7, color='skyblue', edgecolor='black')
axes[1].set_xticks(range(len(gamma_counts)))
axes[1].set_xticklabels([f'{g:.2f}' for g in gamma_counts.index], rotation=45)
axes[1].set_xlabel('Gamma Value', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Distribution of Gamma Values', fontsize=14)
axes[1].grid(True, alpha=0.3)

# Plot 3: Distribution of T values
T_counts = param_df['T'].value_counts().sort_index()
axes[2].bar(range(len(T_counts)), T_counts.values, 
           alpha=0.7, color='lightcoral', edgecolor='black')
axes[2].set_xticks(range(len(T_counts)))
axes[2].set_xticklabels([f'{t}' for t in T_counts.index], rotation=45)
axes[2].set_xlabel('T Value', fontsize=12)
axes[2].set_ylabel('Count', fontsize=12)
axes[2].set_title('Distribution of T Values', fontsize=14)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Setup Test Data

Load test data for comprehensive analysis.

In [ ]:
def create_test_loader(batch_size=64, data_dir='./data'):
    """Create test data loader for CIFAR10."""
    transform_test = transforms.Compose([
        transforms.ToTensor(),
    ])

    test_dataset = CIFAR10(
        root=data_dir, 
        train=False, 
        download=True, 
        transform=transform_test
    )

    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=2,
        pin_memory=True
    )
    
    return test_loader

# Load test data
test_loader = create_test_loader(batch_size=64, data_dir=str(project_root / 'data'))

# Get sample inputs for analysis
test_batch = next(iter(test_loader))
test_input, test_labels = test_batch
test_input = test_input.to(device)

# Use first sample for detailed analysis
sample_input = test_input[0:1]  # Single sample
sample_batch = test_input[0:8]  # Small batch for batch analysis

print(f"Sample input shape: {sample_input.shape}")
print(f"Sample batch shape: {sample_batch.shape}")
print(f"Sample label: {test_labels[0].item()}")

## Comprehensive Energy Dynamics Analysis

Analyze energy dynamics across all 15 models and multiple layers.

In [ ]:
# Extract energy dynamics for all models and layers
energy_dynamics_data = {}

for name, model_data in loaded_models.items():
    print(f"\nAnalyzing energy dynamics for {name}...")
    model = model_data["model"]
    gamma = model_data["gamma"]
    T = model_data["T"]
    case_index = model_data["index"]
    
    layer_dynamics = {}
    
    # Analyze layers 0, 1, 2
    for layer_idx in range(3):
        try:
            analyzer = AKOrNDynamicalAnalyzer(model, layer_idx=layer_idx, device=device)
            energy_data = analyzer.extract_energy_dynamics(sample_input)
            
            # Check if energy_data contains data for this layer
            if energy_data and layer_idx in energy_data:
                trajectory = energy_data[layer_idx]['trajectory']
                layer_dynamics[layer_idx] = {
                    'trajectory': trajectory,
                    'final_energy': trajectory[-1],
                    'initial_energy': trajectory[0],
                    'energy_change': trajectory[-1] - trajectory[0],
                    'max_energy': max(trajectory),
                    'min_energy': min(trajectory)
                }
                print(f"  Layer {layer_idx}: Final energy = {trajectory[-1]:.4f}")
            else:
                print(f"  Layer {layer_idx}: No energy data available")
                
        except Exception as e:
            print(f"  Layer {layer_idx}: Error - {e}")
    
    energy_dynamics_data[name] = {
        'layers': layer_dynamics,
        'gamma': gamma,
        'T': T,
        'case_index': case_index
    }

print(f"\nCompleted energy dynamics analysis for {len(energy_dynamics_data)} models")

In [ ]:
# Access the first loaded model by name (e.g., "Case 0")
name = "Case 0"
model = loaded_models[name]["model"]
gamma = loaded_models[name]["gamma"]
T     = loaded_models[name]["T"]
case_index = loaded_models[name]["index"]


In [ ]:
analyzer = AKOrNDynamicalAnalyzer(model=model,layer_idx = 1)

In [ ]:

with torch.no_grad():
    # Get intermediate states and energies for all layers
    _, _, xs, es = analyzer.model.feature(sample_input)

In [ ]:
layer_energies = es[0]
energy_values = [float(e.item()) for e in layer_energies]
print(energy_values)

In [ ]:
layer_indices = [0, 1]
# If es is a list, you cannot use advanced indexing like es[layer_indices, :]
# Instead, use a list comprehension to get the elements for each index
selected_es = [es[i] for i in layer_indices]
print(selected_es)

In [ ]:
# Comprehensive energy dynamics visualization
fig, axes = plt.subplots(3, 5, figsize=(25, 15))

# Plot energy trajectories for all cases, organized by layer
for layer_idx in range(3):
    plot_count = 0
    
    for name, data in energy_dynamics_data.items():
        if layer_idx in data['layers'] and plot_count < 5:
            col = plot_count
            row = layer_idx
            
            trajectory = data['layers'][layer_idx]['trajectory']
            gamma = data['gamma']
            T = data['T']
            case_index = data['case_index']
            
            axes[row, col].plot(trajectory, linewidth=2, color=plt.cm.viridis(case_index/14))
            axes[row, col].set_title(f'Case {case_index} (γ={gamma}, T={T})\nLayer {layer_idx}', fontsize=10)
            axes[row, col].set_xlabel('Time Step', fontsize=9)
            axes[row, col].set_ylabel('Energy', fontsize=9)
            axes[row, col].grid(True, alpha=0.3)
            
            # Add final energy annotation
            final_energy = trajectory[-1]
            axes[row, col].text(len(trajectory)-1, final_energy, 
                               f'{final_energy:.3f}', 
                               fontsize=8, ha='left', va='bottom')
            
            plot_count += 1
    
    # Hide unused subplots in this row
    for col in range(plot_count, 5):
        axes[layer_idx, col].set_visible(False)

plt.suptitle('Energy Dynamics Across All Cases and Layers', fontsize=16)
plt.tight_layout()
plt.show()

## Comparative Analysis: All Cases

Compare final energies and dynamics patterns across all 15 cases.

In [ ]:
# Create comprehensive comparison data
comparison_data = []

for name, data in energy_dynamics_data.items():
    gamma = data['gamma']
    T = data['T']
    case_index = data['case_index']
    
    for layer_idx, layer_data in data['layers'].items():
        comparison_data.append({
            'case': name,
            'case_index': case_index,
            'gamma': gamma,
            'T': T,
            'layer': layer_idx,
            'final_energy': layer_data['final_energy'],
            'initial_energy': layer_data['initial_energy'],
            'energy_change': layer_data['energy_change'],
            'max_energy': layer_data['max_energy'],
            'min_energy': layer_data['min_energy']
        })

# Convert to DataFrame
comparison_df = pd.DataFrame(comparison_data)
print(f"Created comparison DataFrame with {len(comparison_df)} entries")
print("\nSample of comparison data:")
print(comparison_df.head(10))

In [ ]:
# Comprehensive comparison visualization
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Plot 1: Final energy vs Gamma (colored by T)
for layer in [0, 1, 2]:
    layer_data = comparison_df[comparison_df['layer'] == layer]
    scatter = axes[0, 0].scatter(layer_data['gamma'], layer_data['final_energy'], 
                                c=layer_data['T'], cmap='viridis', 
                                s=60, alpha=0.7, label=f'Layer {layer}')
axes[0, 0].set_xlabel('Gamma', fontsize=12)
axes[0, 0].set_ylabel('Final Energy', fontsize=12)
axes[0, 0].set_title('Final Energy vs Gamma (colored by T)', fontsize=14)
axes[0, 0].set_xscale('log')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
cbar1 = plt.colorbar(scatter, ax=axes[0, 0])
cbar1.set_label('T Value')

# Plot 2: Final energy vs T (colored by Gamma)
scatter2 = axes[0, 1].scatter(comparison_df['T'], comparison_df['final_energy'], 
                             c=comparison_df['gamma'], cmap='plasma', 
                             s=60, alpha=0.7)
axes[0, 1].set_xlabel('T', fontsize=12)
axes[0, 1].set_ylabel('Final Energy', fontsize=12)
axes[0, 1].set_title('Final Energy vs T (colored by Gamma)', fontsize=14)
axes[0, 1].grid(True, alpha=0.3)
cbar2 = plt.colorbar(scatter2, ax=axes[0, 1])
cbar2.set_label('Gamma Value')

# Plot 3: Energy change vs Case index
for layer in [0, 1, 2]:
    layer_data = comparison_df[comparison_df['layer'] == layer]
    axes[0, 2].scatter(layer_data['case_index'], layer_data['energy_change'], 
                      s=60, alpha=0.7, label=f'Layer {layer}')
axes[0, 2].set_xlabel('Case Index', fontsize=12)
axes[0, 2].set_ylabel('Energy Change', fontsize=12)
axes[0, 2].set_title('Energy Change vs Case Index', fontsize=14)
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Plot 4: Distribution of final energies by layer
for layer in [0, 1, 2]:
    layer_data = comparison_df[comparison_df['layer'] == layer]
    axes[1, 0].hist(layer_data['final_energy'], bins=15, alpha=0.7, 
                   label=f'Layer {layer}', density=True)
axes[1, 0].set_xlabel('Final Energy', fontsize=12)
axes[1, 0].set_ylabel('Density', fontsize=12)
axes[1, 0].set_title('Distribution of Final Energies', fontsize=14)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 5: Gamma-T parameter space heatmap (average final energy)
pivot_data = comparison_df.groupby(['gamma', 'T'])['final_energy'].mean().reset_index()
pivot_table = pivot_data.pivot(index='gamma', columns='T', values='final_energy')
im = axes[1, 1].imshow(pivot_table.values, cmap='coolwarm', aspect='auto')
axes[1, 1].set_xticks(range(len(pivot_table.columns)))
axes[1, 1].set_yticks(range(len(pivot_table.index)))
axes[1, 1].set_xticklabels(pivot_table.columns)
axes[1, 1].set_yticklabels([f'{g:.2f}' for g in pivot_table.index])
axes[1, 1].set_xlabel('T', fontsize=12)
axes[1, 1].set_ylabel('Gamma', fontsize=12)
axes[1, 1].set_title('Average Final Energy\n(Gamma-T Space)', fontsize=14)
cbar3 = plt.colorbar(im, ax=axes[1, 1])
cbar3.set_label('Average Final Energy')

# Plot 6: Energy range (max - min) analysis
comparison_df['energy_range'] = comparison_df['max_energy'] - comparison_df['min_energy']
scatter3 = axes[1, 2].scatter(comparison_df['gamma'], comparison_df['energy_range'], 
                             c=comparison_df['T'], cmap='viridis', 
                             s=60, alpha=0.7)
axes[1, 2].set_xlabel('Gamma', fontsize=12)
axes[1, 2].set_ylabel('Energy Range (Max - Min)', fontsize=12)
axes[1, 2].set_title('Energy Range vs Gamma', fontsize=14)
axes[1, 2].set_xscale('log')
axes[1, 2].grid(True, alpha=0.3)
cbar4 = plt.colorbar(scatter3, ax=axes[1, 2])
cbar4.set_label('T Value')

plt.tight_layout()
plt.show()

## Model Performance Evaluation

Evaluate the classification performance of all 15 models.

In [ ]:
# Evaluate model accuracies
print("Evaluating model accuracies on test set...")
accuracy_results = {}
performance_data = []

for name, model_data in loaded_models.items():
    print(f"\nEvaluating {name}...")
    model = model_data["model"]
    gamma = model_data["gamma"]
    T = model_data["T"]
    case_index = model_data["index"]
    
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(test_loader):
            if batch_idx >= 20:  # Limit to first 20 batches for speed
                break
                
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
            total += target.size(0)
    
    accuracy = correct / total
    accuracy_results[name] = accuracy
    
    performance_data.append({
        'case': name,
        'case_index': case_index,
        'gamma': gamma,
        'T': T,
        'accuracy': accuracy
    })
    
    print(f"  Accuracy: {accuracy*100:.2f}%")

# Create performance DataFrame
performance_df = pd.DataFrame(performance_data)
print(f"\nCompleted performance evaluation for {len(performance_df)} models")
print("\nPerformance Summary:")
print(performance_df.sort_values('accuracy', ascending=False).to_string(index=False))

## Performance vs Dynamical Properties Analysis

Analyze the relationship between model performance and dynamical properties.

In [ ]:
# Merge performance data with energy dynamics data
layer0_dynamics = comparison_df[comparison_df['layer'] == 0][['case', 'case_index', 'gamma', 'T', 'final_energy', 'energy_change']]
merged_data = pd.merge(performance_df, layer0_dynamics, on=['case', 'case_index', 'gamma', 'T'])

print("Merged performance and dynamics data:")
print(merged_data.head())

# Performance vs dynamics visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Accuracy vs Final Energy
scatter1 = axes[0, 0].scatter(merged_data['final_energy'], merged_data['accuracy'], 
                             c=merged_data['gamma'], cmap='viridis', 
                             s=100, alpha=0.7, edgecolors='black')
axes[0, 0].set_xlabel('Final Energy (Layer 0)', fontsize=12)
axes[0, 0].set_ylabel('Test Accuracy', fontsize=12)
axes[0, 0].set_title('Accuracy vs Final Energy', fontsize=14)
axes[0, 0].grid(True, alpha=0.3)
cbar1 = plt.colorbar(scatter1, ax=axes[0, 0])
cbar1.set_label('Gamma')

# Add case index annotations
for _, row in merged_data.iterrows():
    axes[0, 0].annotate(f"{row['case_index']}", 
                       (row['final_energy'], row['accuracy']), 
                       xytext=(3, 3), textcoords='offset points', 
                       fontsize=8, alpha=0.8)

# Plot 2: Accuracy vs Gamma
scatter2 = axes[0, 1].scatter(merged_data['gamma'], merged_data['accuracy'], 
                             c=merged_data['T'], cmap='plasma', 
                             s=100, alpha=0.7, edgecolors='black')
axes[0, 1].set_xlabel('Gamma', fontsize=12)
axes[0, 1].set_ylabel('Test Accuracy', fontsize=12)
axes[0, 1].set_title('Accuracy vs Gamma', fontsize=14)
axes[0, 1].set_xscale('log')
axes[0, 1].grid(True, alpha=0.3)
cbar2 = plt.colorbar(scatter2, ax=axes[0, 1])
cbar2.set_label('T Value')

# Plot 3: Accuracy vs T
scatter3 = axes[1, 0].scatter(merged_data['T'], merged_data['accuracy'], 
                             c=merged_data['gamma'], cmap='viridis', 
                             s=100, alpha=0.7, edgecolors='black')
axes[1, 0].set_xlabel('T', fontsize=12)
axes[1, 0].set_ylabel('Test Accuracy', fontsize=12)
axes[1, 0].set_title('Accuracy vs T', fontsize=14)
axes[1, 0].grid(True, alpha=0.3)
cbar3 = plt.colorbar(scatter3, ax=axes[1, 0])
cbar3.set_label('Gamma')

# Plot 4: Accuracy vs Energy Change
scatter4 = axes[1, 1].scatter(merged_data['energy_change'], merged_data['accuracy'], 
                             c=merged_data['case_index'], cmap='tab10', 
                             s=100, alpha=0.7, edgecolors='black')
axes[1, 1].set_xlabel('Energy Change (Layer 0)', fontsize=12)
axes[1, 1].set_ylabel('Test Accuracy', fontsize=12)
axes[1, 1].set_title('Accuracy vs Energy Change', fontsize=14)
axes[1, 1].grid(True, alpha=0.3)
cbar4 = plt.colorbar(scatter4, ax=axes[1, 1])
cbar4.set_label('Case Index')

plt.tight_layout()
plt.show()

# Compute correlations
print("\nCorrelations between performance and dynamical properties:")
correlations = {
    'Accuracy vs Final Energy': merged_data['accuracy'].corr(merged_data['final_energy']),
    'Accuracy vs Energy Change': merged_data['accuracy'].corr(merged_data['energy_change']),
    'Accuracy vs Gamma': merged_data['accuracy'].corr(merged_data['gamma']),
    'Accuracy vs T': merged_data['accuracy'].corr(merged_data['T'])
}

for corr_name, corr_value in correlations.items():
    print(f"  {corr_name}: {corr_value:.3f}")

## Comprehensive T-Value Analysis

Analyze how different T values affect dynamics across all models.

In [ ]:
# Select representative models for T-value analysis
representative_models = {
    'Low Gamma': None,
    'Medium Gamma': None,
    'High Gamma': None
}

# Find representative models
gamma_values = sorted(performance_df['gamma'].unique())
for name, model_data in loaded_models.items():
    gamma = model_data['gamma']
    if gamma == gamma_values[0] and representative_models['Low Gamma'] is None:
        representative_models['Low Gamma'] = (name, model_data)
    elif gamma == gamma_values[len(gamma_values)//2] and representative_models['Medium Gamma'] is None:
        representative_models['Medium Gamma'] = (name, model_data)
    elif gamma == gamma_values[-1] and representative_models['High Gamma'] is None:
        representative_models['High Gamma'] = (name, model_data)

print("Selected representative models for T-value analysis:")
for category, (name, model_data) in representative_models.items():
    if name:
        print(f"  {category}: {name} (γ={model_data['gamma']}, T={model_data['T']})")

# T-value comparison analysis
T_values_for_analysis = [4, 8, 12, 16, 20]
t_analysis_results = {}

for category, (name, model_data) in representative_models.items():
    if name:
        print(f"\nAnalyzing T-values for {category} ({name})...")
        model = model_data['model']
        
        # Analyze layer 0
        analyzer = AKOrNDynamicalAnalyzer(model, layer_idx=0, device=device)
        
        try:
            comparison_results = analyzer.compare_T_values(sample_input, T_values_for_analysis)
            t_analysis_results[category] = comparison_results
            print(f"  Successfully analyzed {len(T_values_for_analysis)} T values")
            
        except Exception as e:
            print(f"  Error in T-value analysis: {e}")

# Visualize T-value analysis results
if t_analysis_results:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Final energies vs T for different gamma categories
    for category, results in t_analysis_results.items():
        if results:
            final_energies = [data['final_energy'] for data in results]
            axes[0, 0].plot(T_values_for_analysis, final_energies, 
                           marker='o', linewidth=2, label=category)
    
    axes[0, 0].set_xlabel('T Value', fontsize=12)
    axes[0, 0].set_ylabel('Final Energy', fontsize=12)
    axes[0, 0].set_title('Final Energy vs T Value', fontsize=14)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Convergence rates vs T
    for category, results in t_analysis_results.items():
        if results:
            convergence_rates = [data.get('converged', False) for data in results]
            convergence_rates = [sum(convergence_rates[:i+1])/(i+1) for i in range(len(convergence_rates))]
            axes[0, 1].plot(T_values_for_analysis, convergence_rates, 
                           marker='s', linewidth=2, label=category)
    
    axes[0, 1].set_xlabel('T Value', fontsize=12)
    axes[0, 1].set_ylabel('Convergence Rate', fontsize=12)
    axes[0, 1].set_title('Convergence Rate vs T Value', fontsize=14)
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].set_ylim(0, 1.1)
    
    # Plot 3: Energy trajectories for different T values (Low Gamma)
    if 'Low Gamma' in t_analysis_results and t_analysis_results['Low Gamma']:
        for i, (T_val, data) in enumerate(zip(T_values_for_analysis, t_analysis_results['Low Gamma'])):
            if 'trajectory' in data:
                axes[1, 0].plot(data['trajectory'], alpha=0.7, 
                               label=f'T={T_val}', linewidth=2)
        
        axes[1, 0].set_xlabel('Time Step', fontsize=12)
        axes[1, 0].set_ylabel('Energy', fontsize=12)
        axes[1, 0].set_title('Energy Trajectories (Low Gamma)', fontsize=14)
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
    
    # Plot 4: Energy trajectories for different T values (High Gamma)
    if 'High Gamma' in t_analysis_results and t_analysis_results['High Gamma']:
        for i, (T_val, data) in enumerate(zip(T_values_for_analysis, t_analysis_results['High Gamma'])):
            if 'trajectory' in data:
                axes[1, 1].plot(data['trajectory'], alpha=0.7, 
                               label=f'T={T_val}', linewidth=2)
        
        axes[1, 1].set_xlabel('Time Step', fontsize=12)
        axes[1, 1].set_ylabel('Energy', fontsize=12)
        axes[1, 1].set_title('Energy Trajectories (High Gamma)', fontsize=14)
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Connectivity Decomposition Analysis

Analyze connectivity decomposition for all models using network analysis.

In [ ]:
def extract_connectivity_decomposition(model, layer_idx=0):
    """Extract connectivity decomposition results for network analysis."""
    try:
        # Create static analyzer for the specified layer
        static_analyzer = AKOrNStaticAnalyzer(model, layer_idx, device=device)
        
        # Extract connectivity blocks
        connectivity_blocks = static_analyzer.extract_connectivity_blocks()
        if connectivity_blocks is None:
            return None
        
        # Compute decomposition metrics
        results = {}
        
        # 1. Frobenius norms
        frob_norms = np.linalg.norm(connectivity_blocks, axis=(1, 2))
        results['frob_norms'] = frob_norms
        
        # 2. Rotation/Symmetric decomposition
        c_R, c_S, alpha, beta = static_analyzer.decompose_rotation_symmetric(connectivity_blocks)
        results['c_R'] = c_R
        results['c_S'] = c_S
        results['alpha'] = alpha
        results['beta'] = beta
        
        # 3. Symmetric/Skew-symmetric decomposition  
        p1, p2, p3, q, sym_frob, skew_frob = static_analyzer.decompose_symmetric_skew(connectivity_blocks)
        results['sym_frob'] = sym_frob
        results['skew_frob'] = skew_frob
        
        return results
        
    except Exception as e:
        print(f"Error in connectivity decomposition: {e}")
        return None

# Extract connectivity decomposition for all models
connectivity_data = {}

for name, model_data in loaded_models.items():
    print(f"\nExtracting connectivity decomposition for {name}...")
    model = model_data["model"]
    gamma = model_data["gamma"]
    T = model_data["T"]
    case_index = model_data["index"]
    
    layer_results = {}
    for layer_idx in range(3):  # Analyze layers 0, 1, 2
        decomp_results = extract_connectivity_decomposition(model, layer_idx)
        if decomp_results is not None:
            layer_results[layer_idx] = decomp_results
            print(f"  Layer {layer_idx}: {len(decomp_results['frob_norms'])} connectivity blocks")
    
    if layer_results:
        connectivity_data[name] = {
            'decomposition': layer_results,
            'gamma': gamma,
            'T': T,
            'case_index': case_index
        }

print(f"\nExtracted connectivity decomposition for {len(connectivity_data)} models")

## Summary Statistics and Comprehensive Report

Generate comprehensive summary statistics across all 15 models.

In [ ]:
# Generate comprehensive summary statistics
summary_stats = {
    'parameter_space': {
        'total_models': len(loaded_models),
        'gamma_range': [performance_df['gamma'].min(), performance_df['gamma'].max()],
        'T_range': [performance_df['T'].min(), performance_df['T'].max()],
        'gamma_values': sorted(performance_df['gamma'].unique().tolist()),
        'T_values': sorted(performance_df['T'].unique().tolist())
    },
    'performance': {
        'best_accuracy': performance_df['accuracy'].max(),
        'worst_accuracy': performance_df['accuracy'].min(),
        'mean_accuracy': performance_df['accuracy'].mean(),
        'std_accuracy': performance_df['accuracy'].std(),
        'best_model': performance_df.loc[performance_df['accuracy'].idxmax(), 'case'],
        'worst_model': performance_df.loc[performance_df['accuracy'].idxmin(), 'case']
    },
    'energy_dynamics': {
        'mean_final_energy_by_layer': {
            f'layer_{i}': comparison_df[comparison_df['layer'] == i]['final_energy'].mean() 
            for i in range(3)
        },
        'energy_change_statistics': {
            'mean': comparison_df['energy_change'].mean(),
            'std': comparison_df['energy_change'].std(),
            'min': comparison_df['energy_change'].min(),
            'max': comparison_df['energy_change'].max()
        }
    },
    'correlations': correlations
}

# Print comprehensive summary
print("=== COMPREHENSIVE ANALYSIS SUMMARY ===")
print(f"\nAnalyzed {summary_stats['parameter_space']['total_models']} models")
print(f"Gamma range: {summary_stats['parameter_space']['gamma_range']}")
print(f"T range: {summary_stats['parameter_space']['T_range']}")

print("\n=== PERFORMANCE RESULTS ===")
print(f"Best accuracy: {summary_stats['performance']['best_accuracy']*100:.2f}% ({summary_stats['performance']['best_model']})")
print(f"Worst accuracy: {summary_stats['performance']['worst_accuracy']*100:.2f}% ({summary_stats['performance']['worst_model']})")
print(f"Mean accuracy: {summary_stats['performance']['mean_accuracy']*100:.2f}% (±{summary_stats['performance']['std_accuracy']*100:.2f}%)")

print("\n=== ENERGY DYNAMICS ===")
for layer, energy in summary_stats['energy_dynamics']['mean_final_energy_by_layer'].items():
    print(f"Average final energy - {layer}: {energy:.4f}")

print("\n=== KEY CORRELATIONS ===")
for corr_name, corr_value in summary_stats['correlations'].items():
    print(f"{corr_name}: {corr_value:.3f}")

# Top and bottom performers
print("\n=== TOP 3 PERFORMERS ===")
top_performers = performance_df.nlargest(3, 'accuracy')
for _, row in top_performers.iterrows():
    print(f"{row['case']}: {row['accuracy']*100:.2f}% (γ={row['gamma']}, T={row['T']})")

print("\n=== BOTTOM 3 PERFORMERS ===")
bottom_performers = performance_df.nsmallest(3, 'accuracy')
for _, row in bottom_performers.iterrows():
    print(f"{row['case']}: {row['accuracy']*100:.2f}% (γ={row['gamma']}, T={row['T']})")

# Save comprehensive results
results_path = project_root / "results" / "comprehensive_analysis_summary.json"
with open(results_path, 'w') as f:
    json.dump(summary_stats, f, indent=2, default=str)

print(f"\nComprehensive analysis results saved to: {results_path}")

## Final Comprehensive Visualization

Create a final comprehensive visualization summarizing all findings.

In [ ]:
# Create final comprehensive visualization
fig = plt.figure(figsize=(24, 18))
gs = fig.add_gridspec(4, 4, hspace=0.3, wspace=0.3)

# Plot 1: Parameter space with performance coloring
ax1 = fig.add_subplot(gs[0, 0:2])
scatter = ax1.scatter(performance_df['gamma'], performance_df['T'], 
                     c=performance_df['accuracy'], cmap='RdYlBu', 
                     s=150, alpha=0.8, edgecolors='black', linewidth=1)
ax1.set_xlabel('Gamma', fontsize=12)
ax1.set_ylabel('T', fontsize=12)
ax1.set_title('Parameter Space: Performance Overview', fontsize=14)
ax1.set_xscale('log')
ax1.grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=ax1)
cbar.set_label('Test Accuracy', fontsize=12)

# Add case annotations
for _, row in performance_df.iterrows():
    ax1.annotate(f"{row['case_index']}", 
                (row['gamma'], row['T']), 
                xytext=(0, 0), textcoords='offset points', 
                ha='center', va='center', fontsize=10, 
                color='white', weight='bold')

# Plot 2: Performance ranking
ax2 = fig.add_subplot(gs[0, 2:4])
sorted_perf = performance_df.sort_values('accuracy', ascending=True)
bars = ax2.barh(range(len(sorted_perf)), sorted_perf['accuracy'], 
                color=plt.cm.RdYlBu(sorted_perf['accuracy']), 
                alpha=0.8, edgecolor='black')
ax2.set_yticks(range(len(sorted_perf)))
ax2.set_yticklabels([f"Case {idx}" for idx in sorted_perf['case_index']])
ax2.set_xlabel('Test Accuracy', fontsize=12)
ax2.set_title('Performance Ranking (All Models)', fontsize=14)
ax2.grid(True, alpha=0.3, axis='x')

# Plot 3: Energy dynamics by layer
ax3 = fig.add_subplot(gs[1, 0:2])
for layer in [0, 1, 2]:
    layer_data = comparison_df[comparison_df['layer'] == layer]
    ax3.scatter(layer_data['gamma'], layer_data['final_energy'], 
               alpha=0.6, s=60, label=f'Layer {layer}')
ax3.set_xlabel('Gamma', fontsize=12)
ax3.set_ylabel('Final Energy', fontsize=12)
ax3.set_title('Final Energy vs Gamma (All Layers)', fontsize=14)
ax3.set_xscale('log')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Performance vs dynamics correlation
ax4 = fig.add_subplot(gs[1, 2:4])
ax4.scatter(merged_data['final_energy'], merged_data['accuracy'], 
           c=merged_data['gamma'], cmap='viridis', 
           s=120, alpha=0.7, edgecolors='black')
ax4.set_xlabel('Final Energy (Layer 0)', fontsize=12)
ax4.set_ylabel('Test Accuracy', fontsize=12)
ax4.set_title('Performance vs Energy Dynamics', fontsize=14)
ax4.grid(True, alpha=0.3)

# Add correlation line
z = np.polyfit(merged_data['final_energy'], merged_data['accuracy'], 1)
p = np.poly1d(z)
ax4.plot(merged_data['final_energy'], p(merged_data['final_energy']), 
         "r--", alpha=0.8, linewidth=2)
ax4.text(0.05, 0.95, f'r = {correlations["Accuracy vs Final Energy"]:.3f}', 
         transform=ax4.transAxes, fontsize=12, 
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Plot 5: Energy trajectory comparison (representative cases)
ax5 = fig.add_subplot(gs[2, 0:2])
representative_cases = [0, 7, 14]  # Low, medium, high case indices
for case_idx in representative_cases:
    case_name = f"Case {case_idx}"
    if case_name in energy_dynamics_data:
        data = energy_dynamics_data[case_name]
        if 0 in data['layers']:
            trajectory = data['layers'][0]['trajectory']
            gamma = data['gamma']
            T = data['T']
            ax5.plot(trajectory, linewidth=2, alpha=0.8, 
                    label=f'Case {case_idx} (γ={gamma}, T={T})')

ax5.set_xlabel('Time Step', fontsize=12)
ax5.set_ylabel('Energy', fontsize=12)
ax5.set_title('Energy Trajectories (Representative Cases)', fontsize=14)
ax5.legend()
ax5.grid(True, alpha=0.3)

# Plot 6: Parameter distributions
ax6 = fig.add_subplot(gs[2, 2:4])
gamma_unique = sorted(performance_df['gamma'].unique())
T_unique = sorted(performance_df['T'].unique())

# Create 2D histogram for parameter combinations
gamma_indices = [gamma_unique.index(g) for g in performance_df['gamma']]
T_indices = [T_unique.index(t) for t in performance_df['T']]
accuracies = performance_df['accuracy'].values

# Create matrix for heatmap
heatmap_data = np.zeros((len(gamma_unique), len(T_unique)))
for gi, ti, acc in zip(gamma_indices, T_indices, accuracies):
    heatmap_data[gi, ti] = acc

im = ax6.imshow(heatmap_data, cmap='RdYlBu', aspect='auto')
ax6.set_xticks(range(len(T_unique)))
ax6.set_yticks(range(len(gamma_unique)))
ax6.set_xticklabels(T_unique)
ax6.set_yticklabels([f'{g:.2f}' for g in gamma_unique])
ax6.set_xlabel('T Value', fontsize=12)
ax6.set_ylabel('Gamma Value', fontsize=12)
ax6.set_title('Performance Heatmap (γ-T Space)', fontsize=14)
cbar2 = plt.colorbar(im, ax=ax6)
cbar2.set_label('Accuracy', fontsize=12)

# Plot 7: Statistical summary
ax7 = fig.add_subplot(gs[3, 0:2])
ax7.axis('off')

summary_text = f"""
COMPREHENSIVE ANALYSIS SUMMARY
================================

Models Analyzed: {len(loaded_models)}
Parameter Space: γ ∈ {summary_stats['parameter_space']['gamma_range']}, T ∈ {summary_stats['parameter_space']['T_range']}

Performance Statistics:
• Best: {summary_stats['performance']['best_accuracy']*100:.1f}% ({summary_stats['performance']['best_model']})
• Worst: {summary_stats['performance']['worst_accuracy']*100:.1f}% ({summary_stats['performance']['worst_model']})
• Mean: {summary_stats['performance']['mean_accuracy']*100:.1f}% (±{summary_stats['performance']['std_accuracy']*100:.1f}%)

Key Findings:
• Accuracy vs Final Energy: r = {correlations['Accuracy vs Final Energy']:.3f}
• Accuracy vs Gamma: r = {correlations['Accuracy vs Gamma']:.3f}
• Accuracy vs T: r = {correlations['Accuracy vs T']:.3f}

Energy Dynamics:
• Layer 0 avg final energy: {summary_stats['energy_dynamics']['mean_final_energy_by_layer']['layer_0']:.4f}
• Layer 1 avg final energy: {summary_stats['energy_dynamics']['mean_final_energy_by_layer']['layer_1']:.4f}
• Layer 2 avg final energy: {summary_stats['energy_dynamics']['mean_final_energy_by_layer']['layer_2']:.4f}
"""

ax7.text(0.05, 0.95, summary_text, transform=ax7.transAxes, 
         fontsize=11, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))

# Plot 8: Best vs worst model comparison
ax8 = fig.add_subplot(gs[3, 2:4])
best_case = summary_stats['performance']['best_model']
worst_case = summary_stats['performance']['worst_model']

if best_case in energy_dynamics_data and worst_case in energy_dynamics_data:
    best_data = energy_dynamics_data[best_case]
    worst_data = energy_dynamics_data[worst_case]
    
    if 0 in best_data['layers'] and 0 in worst_data['layers']:
        best_trajectory = best_data['layers'][0]['trajectory']
        worst_trajectory = worst_data['layers'][0]['trajectory']
        
        ax8.plot(best_trajectory, linewidth=3, alpha=0.8, 
                label=f'Best: {best_case} ({best_data["gamma"]}, {best_data["T"]})', 
                color='green')
        ax8.plot(worst_trajectory, linewidth=3, alpha=0.8, 
                label=f'Worst: {worst_case} ({worst_data["gamma"]}, {worst_data["T"]})', 
                color='red')
        
        ax8.set_xlabel('Time Step', fontsize=12)
        ax8.set_ylabel('Energy', fontsize=12)
        ax8.set_title('Best vs Worst Performer\n(Energy Dynamics)', fontsize=14)
        ax8.legend()
        ax8.grid(True, alpha=0.3)

plt.suptitle('Comprehensive AKOrN Analysis: All 15 Models (opbs_0 to opbs_14)', 
             fontsize=20, y=0.98)
plt.show()

print("\n=== ANALYSIS COMPLETE ===")
print("Comprehensive analysis of all 15 AKOrN models has been completed.")
print("This notebook analyzed energy dynamics, performance, and parameter relationships.")
print(f"Results saved to: {results_path}")